In [35]:
!pip install pandas pandera

import pandas as pd
import pandera as pa
from pandera import Column, DataFrameSchema, Check

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [36]:
df = pd.read_csv('nyc_restaurant_inspections_CLEAN.csv', dtype={'PHONE': 'string'}, encoding='utf-8', low_memory=False)

for col in ['INSPECTION DATE', 'GRADE DATE']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

Loaded 398,302 rows, 22 columns


## Validation Schema

We define rules for each column based on what we know about the dataset and the NYC grading system.

In [37]:
valid_boros = ['MANHATTAN', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND']
valid_grades = ['A', 'B', 'C', 'Z', 'P', 'N', 'Not Yet Graded']
valid_flags = ['Critical', 'Not Critical', 'Not Applicable']

schema = DataFrameSchema({

    "CAMIS": Column(
        pa.String,
        checks=Check.str_matches(r"^\d+$"),
        nullable=False
    ),

    "BORO": Column(
        pa.String,
        checks=Check.isin(valid_boros),
        nullable=True
    ),

    "ZIPCODE": Column(
        pa.String,
        checks=Check.str_matches(r"^\d{5}$"),
        nullable=True
    ),

    "PHONE": Column(
        pa.String,
        checks=Check.str_matches(r"^\d{10}$"),
        nullable=True
    ),

    "INSPECTION DATE": Column(
        pa.DateTime,
        checks=[
            Check(lambda s: s != pd.Timestamp('1900-01-01')),
            Check(lambda s: s <= pd.Timestamp.today())
        ],
        nullable=False
    ),

    "SCORE": Column(
        pa.Float,
        checks=Check.ge(0),
        nullable=True
    ),

    "GRADE": Column(
        pa.String,
        checks=Check.isin(valid_grades),
        nullable=True
    ),

    "CRITICAL FLAG": Column(
        pa.String,
        checks=Check.isin(valid_flags),
        nullable=True
    ),

}, coerce=True)

In [ ]:
# violation code and desc must match nullabilty
@pa.check("VIOLATION CODE", "VIOLATION DESCRIPTION")
def violation_consistency(code, desc):
    return code.isna() == desc.isna()


# HAS_GRADE must match whether GRADE exists or not
@pa.check("HAS_GRADE", "GRADE")
def grade_flag_consistency(flag, grade):
    return flag == grade.notna()

#noo duplicates
@pa.dataframe_check
def no_duplicates(df):
    return ~df.duplicated()

In [39]:
try:
    validated_df = schema.validate(df, lazy=True)
    print("All checks passed!")
except pa.errors.SchemaErrors as e:
    print("Validation failed!")
    print(e.failure_cases)

All checks passed!
